In [ ]:
!pip install ultralytics

In [ ]:
#import libraries and make folder

from pathlib import Path
import cv2
import numpy as np

from ultralytics import YOLO
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from google.colab import files
from IPython.display import display, Image, Video

project_folder = Path("/content/ODOL")
models_folder = project_folder / "models"
results_folder = project_folder / "results" / "pipeline"
image_results_folder = results_folder / "images"
video_results_folder = results_folder / "videos"

models_folder.mkdir(parents=True,exist_ok=True)
image_results_folder.mkdir(parents=True,exist_ok=True)
video_results_folder.mkdir(parents=True,exist_ok=True)

models_folder

In [ ]:
from urllib.request import urlretrieve

urlretrieve("https://github.com/kaiichin/ODOL-Truck-Detection/releases/download/v2.0.0/best.pt",models_folder / "best.pt")
urlretrieve("https://github.com/kaiichin/ODOL-Truck-Detection/releases/download/v2.0.0/mobilenet_odol_classifier.keras",models_folder / "mobilenet_odol_classifier.keras")

models_folder

In [ ]:
yolo_model = YOLO(models_folder / "best.pt")
classifier_model = load_model(models_folder / "mobilenet_odol_classifier.keras")

In [ ]:
detection_threshold = 0.30
not_odol_threshold = 0.40
odol_threshold = 0.60

minimum_crop_size = 80
history_length = 5

prediction_history = {}

def process_frame(frame,tracking=False):
    output_frame = frame.copy()
    frame_width = output_frame.shape[1]

    if tracking:
        results = yolo_model.track(frame,persist=True,tracker="bytetrack.yaml",conf=detection_threshold,imgsz=640,verbose=False)
    else:
        results = yolo_model(frame,conf=detection_threshold,imgsz=640,verbose=False)

    for box in results[0].boxes:
        x1,y1,x2,y2 = box.xyxy[0].cpu().numpy().astype(int)
        detection_confidence = box.conf[0].item()

        if box.id is not None:
            truck_id = int(box.id.item())
            truck_name = f"Truck {truck_id}"
        else:
            truck_id = None
            truck_name = "Truck"

        truck_crop = frame[y1:y2,x1:x2]
        crop_height,crop_width = truck_crop.shape[:2]

        if crop_width < minimum_crop_size or crop_height < minimum_crop_size:
            text = f"{truck_name} | TOO FAR | Detection {detection_confidence:.1%}"
            color = (0,165,255)

        else:
            truck_crop = cv2.cvtColor(truck_crop,cv2.COLOR_BGR2RGB)
            truck_crop = cv2.resize(truck_crop,(224,224))
            truck_crop = np.expand_dims(truck_crop,axis=0)
            truck_crop = preprocess_input(truck_crop.astype(np.float32))

            prediction = classifier_model.predict(truck_crop,verbose=0)[0][0].item()

            if truck_id is not None:
                if truck_id not in prediction_history:
                    prediction_history[truck_id] = []

                prediction_history[truck_id].append(prediction)
                prediction_history[truck_id] = prediction_history[truck_id][-history_length:]

                average_prediction = np.mean(prediction_history[truck_id])
            else:
                average_prediction = prediction

            if average_prediction >= odol_threshold:
                text = f"{truck_name} | ODOL {average_prediction:.1%} | Detection {detection_confidence:.1%}"
                color = (0,0,255)

            elif average_prediction <= not_odol_threshold:
                confidence = 1 - average_prediction
                text = f"{truck_name} | NOT ODOL {confidence:.1%} | Detection {detection_confidence:.1%}"
                color = (0,255,0)

            else:
                text = f"{truck_name} | UNCERTAIN | ODOL score {average_prediction:.1%}"
                color = (0,255,255)

        text_width = cv2.getTextSize(text,cv2.FONT_HERSHEY_SIMPLEX,0.7,2)[0][0]

        text_x = max(0,min(x1,frame_width - text_width - 10))
        text_y = max(y1 - 10,30)

        cv2.rectangle(output_frame,(x1,y1),(x2,y2),color,3)
        cv2.putText(output_frame,text,(text_x,text_y),cv2.FONT_HERSHEY_SIMPLEX,0.7,color,2)

    return output_frame

In [ ]:
uploaded_images = files.upload()

for image_name in uploaded_images:
    image = cv2.imread(image_name)
    result_image = process_frame(image)

    output_path = f"/content/ODOL/results/pipeline/images/{Path(image_name).stem}_result.jpg"

    cv2.imwrite(output_path,result_image)
    display(Image(output_path,width=900))

In [ ]:
# download the image file

files.download(output_path)

In [ ]:
uploaded_videos = files.upload()

video_name = next(iter(uploaded_videos))
output_video_path = "/content/ODOL/results/pipeline/videos/odol_result.mp4"

video = cv2.VideoCapture(video_name)

fps = video.get(cv2.CAP_PROP_FPS)
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video_writer = cv2.VideoWriter(output_video_path,fourcc,fps,(width,height))

In [ ]:
prediction_history.clear()

while True:
    success,frame = video.read()

    if not success:
        break

    result_frame = process_frame(frame,tracking=True)
    video_writer.write(result_frame)

video.release()
video_writer.release()

output_video_path

In [ ]:
!ffmpeg -y -loglevel error -i "/content/ODOL/results/pipeline/videos/odol_result.mp4" -vcodec libx264 -pix_fmt yuv420p "/content/ODOL/results/pipeline/videos/odol_result_playable.mp4"

Video("/content/ODOL/results/pipeline/videos/odol_result_playable.mp4",embed=True,width=900)

In [ ]:
files.download("/content/ODOL/results/pipeline/videos/odol_result_playable.mp4")